# Degraded-Query Eval — BM25 vs Dense on realistic queries

The headline experiment. Runs **both** BM25 and dense retrieval against the
degraded query set (from `query_degradation.ipynb`), using the SAME gold
evidence as the originals. Then compares to the original-query results.

Hypothesis: on degraded (lower-lexical-overlap) queries, BM25 drops more than
dense — the gap closes or flips. If so, that's the finding: naive synthetic
benchmarks over-reward lexical retrieval; realistic phrasing changes the
answer to 'BM25 or dense?'.

**Prereqs:** `rag_eval/` uploaded; `data_processing.ipynb` run through dense
ingestion (so chunks + FAISS index exist in S3); `query_degradation.ipynb`
run (so `data/degraded_queries.parquet` exists).

## 1. Session + deps

In [ ]:
import sys
!{sys.executable} -m pip install -q rank-bm25==0.2.2 faiss-cpu==1.15.1
sys.path.insert(0, ".")
import io, os, tempfile, time
import boto3, numpy as np, pandas as pd, faiss

REGION = "us-east-1"
BUCKET = "rag-arch-eval-data-875699230934"

s3 = boto3.client("s3", region_name=REGION)
def read_parquet_s3(key):
    body = s3.get_object(Bucket=BUCKET, Key=key)["Body"].read()
    return pd.read_parquet(io.BytesIO(body))

## 2. Load degraded queries + chunks + FAISS index

The degraded set carries the original gold `evidence_titles`, so scoring is
apples-to-apples with the original run.

In [ ]:
dq = read_parquet_s3("data/degraded_queries.parquet")
qrecs = [{"query": r.degraded_query, "question_type": r.question_type,
          "rel": set(r.evidence_titles)} for r in dq.itertuples()]
print("degraded queries:", len(qrecs))

from rag_eval.chunking import Chunk
chunks_df = read_parquet_s3("chunks/chunks.parquet")
chunks = [Chunk(r.chunk_id, r.doc_id, r.text) for r in chunks_df.itertuples()]

with tempfile.TemporaryDirectory() as d:
    local = os.path.join(d, "faiss.index")
    s3.download_file(BUCKET, "indexes/faiss.index", local)
    index = faiss.read_index(local)
id_map = read_parquet_s3("indexes/id_map.parquet").sort_values("row")
row_to_doc = id_map["doc_id"].tolist()
print("chunks:", len(chunks), "| index:", index.ntotal)

## 3. Build both retrievers

BM25 over the chunks; dense = Titan query embedding + FAISS. Same metrics,
same fetch depth as the original runs so numbers are comparable.

In [ ]:
from rag_eval.retrievers import BM25Retriever, ranked_doc_ids
from rag_eval.bedrock import TitanEmbedder
from rag_eval.metrics import METRICS

KS = [1, 3, 5, 10, 20]; FETCH = max(KS) * 5

bm25 = BM25Retriever(); bm25.index(chunks)
embedder = TitanEmbedder(model_id="amazon.titan-embed-text-v2:0", region=REGION)

def normalize(m):
    n = np.linalg.norm(m, axis=1, keepdims=True); n[n == 0] = 1.0
    return m / n

def dense_ids(q, fetch):
    v = normalize(embedder.embed_query(q).reshape(1, -1).astype(np.float32))
    _, idx = index.search(v, min(fetch, index.ntotal))
    seen, out = set(), []
    for i in idx[0]:
        if i == -1: continue
        d = row_to_doc[i]
        if d not in seen: seen.add(d); out.append(d)
    return out

## 4. Score both retrievers on the degraded queries

In [ ]:
def score_all(rank_fn, label):
    rows = []
    for j, q in enumerate(qrecs, 1):
        ranked = rank_fn(q["query"], FETCH)
        row = {"question_type": q["question_type"]}
        for k in KS:
            for name, fn in METRICS.items():
                row[f"{name}@{k}"] = fn(ranked, q["rel"], k)
        rows.append(row)
        if j % 500 == 0: print(f"  [{label}] {j}/{len(qrecs)}")
    return pd.DataFrame(rows)

bm25_deg = score_all(lambda q, k: ranked_doc_ids(bm25.retrieve(q, k)), "bm25")
dense_deg = score_all(dense_ids, "dense")
print("scored both retrievers on degraded queries")

## 5. The comparison table

Overall recall@10 / all_hit@10 for BM25 vs dense on degraded queries. Put
these next to your ORIGINAL-query numbers (from bm25_eval / dense_eval). The
story is in the *deltas*: how much each retriever dropped going from
original -> degraded.

In [ ]:
cols = ["recall@5", "recall@10", "all_hit@10", "mrr@10", "ndcg@10"]
summary = pd.DataFrame({
    "bm25_degraded": bm25_deg[cols].mean().round(4),
    "dense_degraded": dense_deg[cols].mean().round(4),
})
print("=== DEGRADED QUERIES: BM25 vs Dense ===")
print(summary.to_string())

print("\n=== DEGRADED, by question type (recall@10) ===")
cmp = pd.DataFrame({
    "bm25": bm25_deg.groupby("question_type")["recall@10"].mean().round(4),
    "dense": dense_deg.groupby("question_type")["recall@10"].mean().round(4),
})
print(cmp.to_string())
print("\nFill in original-query numbers from bm25_eval/dense_eval to complete")
print("the original-vs-degraded x bm25-vs-dense story.")

## 6. Save degraded results to S3

In [ ]:
for df, key in [
    (bm25_deg, "results/notebook/bm25_degraded_per_query.csv"),
    (dense_deg, "results/notebook/dense_degraded_per_query.csv"),
    (summary.reset_index(), "results/notebook/degraded_summary.csv"),
]:
    b = io.StringIO(); df.to_csv(b, index=False)
    s3.put_object(Bucket=BUCKET, Key=key, Body=b.getvalue().encode())
    print("wrote s3://%s/%s" % (BUCKET, key))